# **Deep Learning Pipeline for Time Series Forecasting**

This pipeline automates **time series forecasting** with deep learning models, including **TiDE+RIN, N-Beats, NHiTS, and NLinear**. It preprocesses data, trains models, and evaluates performance using **sliding window forecasting**.

---

## **Pipeline Overview**

### **1. Data Handling**
- Loads **oil production (Volve, UNISIM-II-H) & energy generation (OPSD)** data.
- Cleans missing values, standardizes column names, and indexes time series.
- Converts data to `Darts` `TimeSeries` format.

### **2. Model Training**
- Supports multiple deep learning models:
  - **TiDE, TiDE+RIN, N-Beats, NHiTS, NLinear**
- Trains using **time series covariates** and **early stopping**.
- Sliding window approach for adaptive learning.

### **3. Forecasting & Evaluation**
- Uses **walk-forward validation** with `iterative_forecast_deep_encoder`.
- Measures performance with **Mean Absolute Error (MAE) & Mean Squared Error (MSE)**.
- Aggregates metrics across datasets for robust comparison.

---

## **Dataset Summary**

| Dataset    | Description               | Target Variable                   | Frequency |
|------------|---------------------------|-----------------------------------|------------|
| **Volve**  | Offshore oil production   | `BORE_OIL_VOL`                   | Daily      |
| **UNISIM** | Reservoir simulation      | `QOOB`                            | Daily      |
| **OPSD**   | Energy generation (wind)  | `GB_GBN_wind_generation_actual`  | 30 min     |

---

## **Key Features**
✅ **Automated data preprocessing & feature engineering**  
✅ **Sliding window forecasting for adaptive learning**  
✅ **Deep learning models optimized for long-term predictions**  
✅ **Scalable for various industries & time series applications**  

🚀 **Designed for real-world forecasting at scale!**

In [ ]:
# Standard library imports
import os
import warnings
import logging

from src.darts_common.config_wells import DATA_SOURCES
from src.data.data_loading import DataSource
from src.darts_common.preprocessing import process_data_source

# Disable warnings and logging
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

# Disable CUDA
os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Adjust working directory to project root
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
print(f"Working directory set to: {os.getcwd()}")

In [ ]:
def main_online():
    """
    Função principal para rodar o pipeline de forecasting online com abordagem de sliding window.
    """
    import warnings, logging
    warnings.filterwarnings("ignore")
    logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
    
    # Seleciona os pipelines desejados
    selected_names = ["VOLVE", "UNISIM", "OPSD"]
    filtered_configs = [cfg for cfg in DATA_SOURCES if cfg["name"] in selected_names]
    
    # Parâmetros do forecast (ajuste conforme necessário)
    lag_window = 7             
    forecast_horizon = 112     
    initial_train_size = 188 + forecast_horizon
    sampling_rate = 1          
    model_type = "TiDE+RIN"
    
    validation_ratio = 0.6     
    stride = 7                 
    
    all_metrics = []
    
    # Processa cada fonte de dados
    for config in filtered_configs:
        data_source_obj = DataSource(config)
        loader = data_source_obj.get_loader()
        # Carrega os dados usando o loader unificado
        preloaded_data = loader.load()
                
        # Agora, passa os dados pré-carregados para o processamento
        process_data_source(data_source_obj.__dict__, model_type, initial_train_size, 
                              forecast_horizon, sampling_rate, all_metrics, 
                              validation_ratio, stride, preloaded_data=preloaded_data)
                              
if __name__ == "__main__":
    main_online()